# Tahap 3: Penyiapan Tabel Resep di PostgreSQL
Notebook ini menghubungkan ke database PostgreSQL menggunakan konfigurasi dari .env, membuat tabel recipes jika belum ada, dan melakukan upsert data dari cleaned_recipes.csv.

In [1]:
import os
import sys
import pandas as pd
import psycopg2
from psycopg2.extras import execute_batch
from dotenv import load_dotenv

if sys.stdout.encoding != 'utf-8':
    try:
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
    except Exception:
        pass

# Load environment variables dari .env
load_dotenv(dotenv_path=".env" if os.path.exists(".env") else None)

db_host = os.getenv("DB_HOST", "localhost")
db_port = os.getenv("DB_PORT", "5432")
db_name = os.getenv("DB_NAME", "dietmate_db")
db_user = os.getenv("DB_USER", "postgres")
db_password = os.getenv("DB_PASSWORD", "bungaalisa")

conn = psycopg2.connect(
    host=db_host,
    port=db_port,
    dbname=db_name,
    user=db_user,
    password=db_password
)
print(f"Terhubung ke PostgreSQL: host={db_host}, port={db_port}, db={db_name}, user={db_user}")

Terhubung ke PostgreSQL: host=localhost, port=5432, db=dietmate_db, user=postgres


In [2]:
create_sql = """
CREATE TABLE IF NOT EXISTS recipes (
    recipe_id VARCHAR(50) PRIMARY KEY,
    nama_masakan VARCHAR(255) NOT NULL,
    kategori VARCHAR(100),
    jumlah_porsi INTEGER DEFAULT 1,
    basis_porsi VARCHAR(50) DEFAULT '1 porsi',
    kalori_kkal NUMERIC(8, 2) DEFAULT 0.0,
    protein_g NUMERIC(8, 2) DEFAULT 0.0,
    karbohidrat_g NUMERIC(8, 2) DEFAULT 0.0,
    lemak_g NUMERIC(8, 2) DEFAULT 0.0,
    bahan_bahan TEXT,
    cara_memasak TEXT,
    url TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""
with conn.cursor() as cur:
    cur.execute(create_sql)
conn.commit()
print("Tabel 'recipes' siap digunakan.")

Tabel 'recipes' siap digunakan.


In [3]:
csv_path = os.path.join("dataset", "cleaned_recipes.csv")
df = pd.read_csv(csv_path)
print(f"Membaca {len(df)} baris data dari {csv_path}")

upsert_sql = """
INSERT INTO recipes (
    recipe_id, nama_masakan, kategori, jumlah_porsi, basis_porsi,
    kalori_kkal, protein_g, karbohidrat_g, lemak_g,
    bahan_bahan, cara_memasak, url
) VALUES (
    %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s
)
ON CONFLICT (recipe_id) DO UPDATE SET
    nama_masakan = EXCLUDED.nama_masakan,
    kategori = EXCLUDED.kategori,
    jumlah_porsi = EXCLUDED.jumlah_porsi,
    basis_porsi = EXCLUDED.basis_porsi,
    kalori_kkal = EXCLUDED.kalori_kkal,
    protein_g = EXCLUDED.protein_g,
    karbohidrat_g = EXCLUDED.karbohidrat_g,
    lemak_g = EXCLUDED.lemak_g,
    bahan_bahan = EXCLUDED.bahan_bahan,
    cara_memasak = EXCLUDED.cara_memasak,
    url = EXCLUDED.url;
"""

records = []
for _, row in df.iterrows():
    records.append((
        str(row['recipe_id']),
        str(row['nama_masakan']),
        str(row['kategori']) if pd.notna(row['kategori']) else None,
        int(row['jumlah_porsi']) if pd.notna(row['jumlah_porsi']) else 1,
        str(row['basis_porsi']) if pd.notna(row['basis_porsi']) else '1 porsi',
        float(row['kalori_kkal']) if pd.notna(row['kalori_kkal']) else 0.0,
        float(row['protein_g']) if pd.notna(row['protein_g']) else 0.0,
        float(row['karbohidrat_g']) if pd.notna(row['karbohidrat_g']) else 0.0,
        float(row['lemak_g']) if pd.notna(row['lemak_g']) else 0.0,
        str(row['bahan_bahan']) if pd.notna(row['bahan_bahan']) else "",
        str(row['cara_memasak']) if pd.notna(row['cara_memasak']) else "",
        str(row['url']) if pd.notna(row['url']) else ""
    ))

with conn.cursor() as cur:
    execute_batch(cur, upsert_sql, records, page_size=100)
conn.commit()
print(f"Berhasil melakukan upsert {len(records)} baris ke tabel recipes.")

Membaca 327 baris data dari dataset\cleaned_recipes.csv


Berhasil melakukan upsert 327 baris ke tabel recipes.


In [4]:
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM recipes;")
    total_count = cur.fetchone()[0]
    cur.execute("""
        SELECT recipe_id, nama_masakan, jumlah_porsi, basis_porsi, kalori_kkal, protein_g
        FROM recipes
        ORDER BY recipe_id ASC
        LIMIT 3;
    """)
    samples = cur.fetchall()

print(f"Total baris pada tabel recipes: {total_count}")
for row in samples:
    print(f"- {row}")

conn.close()
print("Koneksi PostgreSQL ditutup.")

Total baris pada tabel recipes: 327
- ('RCP_0001', 'Toge Telur', 1, '1 porsi', Decimal('252.00'), Decimal('21.90'))
- ('RCP_0002', 'Tumis Labu Siam', 5, '1 porsi', Decimal('61.00'), Decimal('0.79'))
- ('RCP_0003', 'Dimsum Ayam Udang', 27, '1 porsi', Decimal('50.00'), Decimal('3.70'))


Koneksi PostgreSQL ditutup.
